In [1]:
import ase
from mace.calculators import MACECalculator

import numpy as np
import matplotlib.pyplot as plt

from ase.build import graphene
from ase.phonons import Phonons

from ase.filters import ExpCellFilter
from ase.optimize import LBFGS

/home/gabrielecolombo/Thesis/PROJECT/new_sources/mace_custom/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/gabrielecolombo/Thesis/PROJECT/new_sources/mace_custom/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
import subprocess
ROOT = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()

model_path = f"{ROOT}/KAGGLE_DOWNLOAD/from_C2_v1/MACE.model"

In [3]:
model = MACECalculator(model_path, device="cuda")

/home/gabrielecolombo/Thesis/PROJECT/new_sources/mace_custom/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/gabrielecolombo/Thesis/PROJECT/new_sources/mace_custom/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


/home/gabrielecolombo/Thesis/PROJECT/new_sources/mace_custom/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/gabrielecolombo/Thesis/PROJECT/new_sources/mace_custom/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


### Relax graphene

In [4]:
graphene_atoms = graphene()
graphene_atoms.cell[2][2] = 100
graphene_atoms.calc = model

filter = ExpCellFilter(graphene_atoms, mask = [1, 1, 0, 0, 0, 0])
opt = LBFGS(filter)

opt.run(1e-5)


/tmp/ipykernel_130765/574277464.py:5: DeprecationWarning: Use FrechetCellFilter for better convergence w.r.t. cell variables.
  filter = ExpCellFilter(graphene_atoms, mask = [1, 1, 0, 0, 0, 0])


       Step     Time          Energy          fmax
LBFGS:    0 17:15:32      -15.917213        0.014588
LBFGS:    1 17:15:33      -15.917213        0.029116
LBFGS:    2 17:15:33      -15.917212        0.000014
LBFGS:    3 17:15:33      -15.917213        0.000039
LBFGS:    4 17:15:33      -15.917213        0.000029
LBFGS:    5 17:15:33      -15.917213        0.000004


np.True_

In [5]:
supercell = graphene_atoms.repeat((8,8,1))
supercell.calc = model

In [6]:
ph = Phonons(
    graphene_atoms,
    model,
    supercell = (8, 8, 1),
    delta=0.01,
)

In [7]:
hessian = model.get_hessian(supercell)

In [8]:
hessian.shape

(384, 128, 3)

In [9]:
hessian_reshaped = hessian.reshape(128*3, 128*3)

In [10]:
point = np.array([1./3., 1./3., 0])
dyn_matrix = ph.compute_dynamical_matrix(point, hessian_reshaped)

In [11]:
DEG2RAD = np.pi / 180.

angle = 75 * DEG2RAD
C = np.cos(angle)
S = np.sin(angle)

strain = np.array([[C, 0, 0], [0, S, 0], [0, 0, 0]])

amplitudes = np.linspace(0, 0.5, 40)

identity = np.eye(3)

deformations = [identity + a*strain for a in amplitudes]

In [12]:
c0 = graphene_atoms.cell.copy()
path = graphene_atoms.cell.bandpath("GMKG", npoints=200)

import os
from tqdm import tqdm

os.makedirs("TMP", exist_ok=True)

for i, d in tqdm(enumerate(deformations)):
    copy = graphene_atoms.copy()
    cell = c0 @ d

    copy.set_cell(cell, scale_atoms = True)
    copy.calc = model

    opt = LBFGS(copy, logfile=None)
    opt.run(1e-5, 1000)

    ph = Phonons(
        copy,
        model,
        supercell = (8, 8, 1),
        delta=0.01,
        name = "cache"
    )

    ph.run()

    ph.read(acoustic=False)
    ph.clean()


    bs = ph.get_band_structure(path, verbose=False)

    emax = bs.energies.max()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    bs.plot(ax=ax, emin=-0.1, emax=emax)

        
    ax.set_title(f"Phonon Dispersion - Deformation Matrix:\n{d}")
    plt.savefig(f"TMP/phonons_{i}.svg")
    
    plt.close(fig) # Important: Clear memory for the next iteration


40it [00:44,  1.12s/it]
